# Reagent Calculator Pubchem

> **Curated from:** `YVR_R1.ipynb`. Original scientific code has not been refactored or validated. Saved execution outputs were cleared; notebook was not re-run. Check the repository README, input paths, atom mapping, dependencies, and author permissions before reuse.


In [ ]:
pip install pubchempy pandas

In [ ]:
import pandas as pd
import pubchempy as pcp
import re
from IPython.display import FileLink

reaction_scale = 0.1  # mmol

reaction_reagents = [
    {"Name": "1-vinylnaphthalene", "Equivalents": 1, "State": "solid", "Density": None},
    {"Name": "morpholine", "Equivalents": 1, "State": "liquid", "Density": 1.00},
    {"Name": "potassium persulfate", "Equivalents": 2, "State": "solid", "Density": None},
    {"Name": "sodium chloride", "Equivalents": 1.5, "State": "solid", "Density": None}
]

data = []

for r in reaction_reagents:

    compound = pcp.get_compounds(r["Name"], 'name')[0]

    mw = compound.molecular_weight

    cas = "Not Found"
    pattern = r"\d{2,7}-\d{2}-\d"

    for syn in compound.synonyms:
        if re.match(pattern, syn):
            cas = syn
            break

    mmol = reaction_scale * r["Equivalents"]

    mass_g = (mw * mmol) / 1000

    volume_ml = ""

    if r["State"] == "liquid" and r["Density"]:
        volume_ml = mass_g / r["Density"]

    data.append({
        "Compound": r["Name"],
        "CAS Number": cas,
        "State": r["State"],
        "MW (g/mol)": mw,
        "Equivalents": r["Equivalents"],
        "mmol": round(mmol,4),
        "Mass (g)": round(mass_g,6),
        "Volume (mL)": round(volume_ml,6) if volume_ml else ""
    })

df = pd.DataFrame(data)

solvent = pd.DataFrame([{
    "Solvent": "Acetonitrile",
    "CAS Number": "75-05-8",
    "Volume (mL)": 1
}])

file_name = "reaction_setup.xlsx"

with pd.ExcelWriter(file_name) as writer:
    df.to_excel(writer, sheet_name="Reagents", index=False)
    solvent.to_excel(writer, sheet_name="Solvent", index=False)

print("Excel file created!")

display(FileLink(file_name))

In [ ]:
df

In [ ]:
temperature = 60   # reaction temperature in °C

procedure = "A reaction vial was charged with "

parts = []

for i,row in df.iterrows():

    if row["Volume (mL)"] != "":
        part = f'{row["Compound"]} ({row["mmol"]} mmol, {row["Volume (mL)"]} mL)'
    else:
        part = f'{row["Compound"]} ({row["mmol"]} mmol, {row["Mass (g)"]} g)'

    parts.append(part)

procedure += ", ".join(parts)

procedure += f". The mixture was dissolved in acetonitrile (1 mL) and stirred at {temperature} °C."

print(procedure)